# Infrastructure as Code with Ansible

Imagine that your team owns a small service that reports its application name, environment, and release version. A teammate changes the deployed version directly during an incident. The service still responds, but nobody can tell whether it matches the configuration in the repository. In this tutorial you will build that service, create this mismatch yourself, and use Ansible to recover a known state.

Run the cells from top to bottom in **Google Colab** or locally on Debian/Ubuntu Linux. In Colab, use [File → Upload notebook](https://colab.research.google.com/) to run this copy; no local installation or cloud provider credentials are needed. A free Google account may be needed. For local execution, run `uv sync --python 3.12` from the repository and select `.venv/bin/python` as the VS Code kernel. Local setup downloads Nginx without administrator rights when it is absent. The README has detailed setup instructions.

> **Running all cells:** The last cell stops the services and deletes the tutorial working directory. If you want to inspect the live service, pause before **Clean up**. After cleanup, rerunning a later verification or drift cell on its own may fail; start again from the setup cell.

## Learning outcomes

After completing the experiments, you should be able to:

1. Explain desired state, convergence, idempotency, and configuration drift using the service you deployed.
2. Read a local Ansible inventory, playbook, variables file, and Jinja templates.
3. Verify a Flask application through its Nginx reverse proxy with automated HTTP assertions.
4. Show that a second playbook run reports `changed=0`.
5. Detect and repair an undocumented change, then perform a reviewed version update.
6. Explain when Ansible is useful and where this single-host example differs from production.

## The system you will build

![Notebook uses Ansible to configure Flask and Nginx; tests call Nginx](assets/architecture.png)

The notebook runs Ansible against `localhost`, so the control node and managed node are the same machine. The HTTP test calls Nginx, which forwards the request to Gunicorn and Flask; this checks the complete request path. Both services listen only on `127.0.0.1`.

## 1. Why declare the desired state?

A manual setup is a sequence of remembered commands. If one command is missed or a file is edited later, two machines can behave differently even when they supposedly run the same release. Ansible instead compares the host with a declaration in files and applies the missing changes. This movement toward the declaration is **convergence**.

The experiment follows one version value through several states:

| Moment | Declared version | Deployed version | Expected result |
| --- | --- | --- | --- |
| First deployment | `1.0` | absent → `1.0` | Service starts |
| Repeat deployment | `1.0` | `1.0` | `changed=0` |
| Manual edit | `1.0` | `manual-edit` | Drift becomes visible |
| Repair | `1.0` | `1.0` | Ansible restores the file |
| Reviewed update | `2.0` | `1.0` → `2.0` | New release is verified |

### Prepare the runtime

The next cell creates a temporary working directory, chooses free local ports, and installs or finds the required tools. On Colab the playbook installs system packages. On a local Debian/Ubuntu host it uses the `uv` environment and extracts Nginx into the working directory if Nginx is absent. **Look for the selected ports in the output**; every later configuration and HTTP check uses them.

In [5]:
import errno
import json
import os
import shutil
import socket
import subprocess
import sys
import tempfile
from pathlib import Path

CAN_INSTALL_OS_PACKAGES = os.geteuid() == 0
WORK = (Path('/content/dd2482-ansible-tutorial') if CAN_INSTALL_OS_PACKAGES
        else Path(tempfile.gettempdir()) / f'dd2482-ansible-tutorial-{os.getuid()}')
DEPLOY = WORK / 'deployed'
PORTS_FILE = WORK / 'ports.json'
WORK.mkdir(parents=True, exist_ok=True)
# A check-mode preview cannot create parent directories. Bootstrap an empty
# directory here; the playbook still declares and manages its state.
DEPLOY.mkdir(parents=True, exist_ok=True)

def pid_is_alive(path):
    try:
        os.kill(int(path.read_text().strip()), 0)
        return True
    except (FileNotFoundError, ValueError, ProcessLookupError):
        return False

def port_is_free(port):
    # bind catches listeners on both 127.0.0.1 and 0.0.0.0.
    with socket.socket() as sock:
        try:
            sock.bind(('127.0.0.1', port))
            return True
        except OSError as error:
            if error.errno == errno.EADDRINUSE:
                return False
            raise

saved_ports = json.loads(PORTS_FILE.read_text()) if PORTS_FILE.exists() else {}

def choose_port(name, preferred, fallback_start, pid_name, excluded=()):
    previous = saved_ports.get(name)
    if (isinstance(previous, int) and 1 <= previous <= 65535
            and previous not in excluded
            and (port_is_free(previous) or pid_is_alive(DEPLOY / pid_name))):
        return previous
    for candidate in (preferred, *range(fallback_start, fallback_start + 100)):
        if candidate not in excluded and port_is_free(candidate):
            return candidate
    raise RuntimeError(f'No free port found for {name}.')

BACKEND_PORT = choose_port('backend', 8000, 38000, 'gunicorn.pid')
PUBLIC_PORT = choose_port('public', 8080, 48000, 'nginx.pid', (BACKEND_PORT,))
PORTS_FILE.write_text(json.dumps({'backend': BACKEND_PORT, 'public': PUBLIC_PORT}) + '\n')
print(f'Backend port: {BACKEND_PORT}; Nginx public port: {PUBLIC_PORT}')
if PUBLIC_PORT != 8080 or BACKEND_PORT != 8000:
    print('A default port was occupied, so free local ports were selected.')

if CAN_INSTALL_OS_PACKAGES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ansible'], check=True)
    APP_VENV = DEPLOY / 'venv'
    NGINX_BINARY = '/usr/sbin/nginx'  # Ansible installs it during the first apply.
    ANSIBLE_PLAYBOOK = shutil.which('ansible-playbook')
else:
    APP_VENV = Path(sys.prefix)
    if sys.prefix == sys.base_prefix or not (APP_VENV / 'bin/gunicorn').exists():
        raise RuntimeError("Run uv sync and select this repository\'s .venv/bin/python kernel.")
    ANSIBLE_PLAYBOOK = str(APP_VENV / 'bin/ansible-playbook')
    NGINX_BINARY = shutil.which('nginx')
    if NGINX_BINARY is None and Path('/usr/sbin/nginx').exists():
        NGINX_BINARY = '/usr/sbin/nginx'
    if NGINX_BINARY is None:
        if not shutil.which('apt-get') or not shutil.which('dpkg-deb'):
            raise RuntimeError('Local execution needs Nginx or apt-get and dpkg-deb (Debian/Ubuntu).')
        package_dir = WORK / 'nginx-package'
        extracted_binary = package_dir / 'usr/sbin/nginx'
        if not extracted_binary.exists():
            subprocess.run(['apt-get', 'download', 'nginx'], cwd=WORK, check=True)
            packages = sorted(WORK.glob('nginx_*.deb'))
            if not packages:
                raise RuntimeError('apt-get did not download an Nginx package.')
            subprocess.run(['dpkg-deb', '-x', str(packages[-1]), str(package_dir)], check=True)
        NGINX_BINARY = str(extracted_binary)

assert ANSIBLE_PLAYBOOK and Path(ANSIBLE_PLAYBOOK).exists()
assert NGINX_BINARY and (CAN_INSTALL_OS_PACKAGES or Path(NGINX_BINARY).exists())
version_commands = [[sys.executable, '--version'], [ANSIBLE_PLAYBOOK, '--version']]
if not CAN_INSTALL_OS_PACKAGES:
    version_commands.append([NGINX_BINARY, '-v'])
for command in version_commands:
    result = subprocess.run(command, text=True, capture_output=True, check=True)
    print((result.stdout or result.stderr).splitlines()[0])
print('Working directory:', WORK)
if CAN_INSTALL_OS_PACKAGES:
    print('Ansible will install Nginx and the application packages.')
else:
    print('Using the uv environment for Ansible, Flask, and Gunicorn.')


Backend port: 8000; Nginx public port: 8080
Python 3.12.3
ansible-playbook [core 2.21.4]
nginx version: nginx/1.24.0 (Ubuntu)
Working directory: /tmp/dd2482-ansible-tutorial-2465940
Using the uv environment for Ansible, Flask, and Gunicorn.


## 2. Inventory and variables

An inventory answers **where should Ansible run?** The single `localhost` entry makes this host both controller and target, avoiding SSH credentials for the exercise. `vars.yml` answers **what should be deployed?** It holds the name, version, environment, and ports. The setup also records paths and whether OS package installation is available.

Run the cell and inspect both files. Find `app_version: "1.0"`: that is the declared release. Later you will change this line for an intentional update, while the drift experiment will edit a different, already deployed file.

In [6]:
import textwrap

def write_source(name, content):
    path = WORK / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding='utf-8')
    print(path)

write_source('inventory.ini', f'''
[tutorial]
localhost ansible_connection=local ansible_python_interpreter={sys.executable}
''')
write_source('vars.yml', f'''
app_name: "dd2482-demo"
app_version: "1.0"
environment_name: "tutorial"
backend_port: {BACKEND_PORT}
public_port: {PUBLIC_PORT}
manage_os_packages: {str(CAN_INSTALL_OS_PACKAGES).lower()}
tutorial_workdir: {json.dumps(str(WORK))}
tutorial_venv: {json.dumps(str(APP_VENV))}
nginx_binary_path: {json.dumps(str(NGINX_BINARY))}
''')
print((WORK / 'inventory.ini').read_text())
print((WORK / 'vars.yml').read_text())


/tmp/dd2482-ansible-tutorial-2465940/inventory.ini
/tmp/dd2482-ansible-tutorial-2465940/vars.yml
[colab]
localhost ansible_connection=local ansible_python_interpreter=/afs/kth.se/home/l/d/ldef/git/DD2482-executable-tutorial/.venv/bin/python

app_name: "dd2482-demo"
app_version: "1.0"
environment_name: "tutorial"
backend_port: 8000
public_port: 8080
manage_os_packages: false
tutorial_workdir: "/tmp/dd2482-ansible-tutorial-2465940"
tutorial_venv: "/afs/kth.se/home/l/d/ldef/git/DD2482-executable-tutorial/.venv"
nginx_binary_path: "/tmp/dd2482-ansible-tutorial-2465940/nginx-package/usr/sbin/nginx"



## 3. Templates for the application and proxy

A Jinja template is a file with placeholders such as `{{ app_version }}`. Ansible will fill those placeholders from the variables when it creates the deployed files. The three templates have different jobs:

- `app.py.j2` defines the Flask route. It reads `config.json` on every request so a file change is visible immediately.
- `config.json.j2` turns the declared name, version, and environment into the response data.
- `nginx.conf.j2` forwards requests to Gunicorn and adds `X-DD2482-Proxy: nginx` to responses.

Run the cell and locate one placeholder in each relevant template. These are still source files; no service has been started yet. That distinction matters when we later edit a deployed file without changing its source template.

In [7]:
write_source('templates/app.py.j2', '''
import json
from pathlib import Path

from flask import Flask, jsonify

app = Flask(__name__)
CONFIG = Path("{{ deploy_dir }}/config.json")

@app.get("/")
def index():
    return jsonify(json.loads(CONFIG.read_text(encoding="utf-8")))
''')
write_source('templates/config.json.j2', '''
{
  "application": "{{ app_name }}",
  "version": "{{ app_version }}",
  "environment": "{{ environment_name }}"
}
''')
write_source('templates/nginx.conf.j2', '''
pid {{ deploy_dir }}/nginx.pid;
error_log {{ deploy_dir }}/nginx-error.log;
events {}
http {
    access_log {{ deploy_dir }}/nginx-access.log;
    client_body_temp_path {{ deploy_dir }}/client_body_temp;
    proxy_temp_path {{ deploy_dir }}/proxy_temp;
    fastcgi_temp_path {{ deploy_dir }}/fastcgi_temp;
    uwsgi_temp_path {{ deploy_dir }}/uwsgi_temp;
    scgi_temp_path {{ deploy_dir }}/scgi_temp;
    server {
        listen 127.0.0.1:{{ public_port }};
        location / {
            proxy_pass http://127.0.0.1:{{ backend_port }};
            add_header X-DD2482-Proxy nginx always;
        }
    }
}
''')
for path in sorted((WORK / 'templates').glob('*')):
    print(f'\n{path.name}:\n{path.read_text()}')


/tmp/dd2482-ansible-tutorial-2465940/templates/app.py.j2
/tmp/dd2482-ansible-tutorial-2465940/templates/config.json.j2
/tmp/dd2482-ansible-tutorial-2465940/templates/nginx.conf.j2

app.py.j2:
import json
from pathlib import Path

from flask import Flask, jsonify

app = Flask(__name__)
CONFIG = Path("{{ deploy_dir }}/config.json")

@app.get("/")
def index():
    return jsonify(json.loads(CONFIG.read_text(encoding="utf-8")))


config.json.j2:
{
  "application": "{{ app_name }}",
  "version": "{{ app_version }}",
  "environment": "{{ environment_name }}"
}


nginx.conf.j2:
pid {{ deploy_dir }}/nginx.pid;
error_log {{ deploy_dir }}/nginx-error.log;
events {}
http {
    access_log {{ deploy_dir }}/nginx-access.log;
    client_body_temp_path {{ deploy_dir }}/client_body_temp;
    proxy_temp_path {{ deploy_dir }}/proxy_temp;
    fastcgi_temp_path {{ deploy_dir }}/fastcgi_temp;
    uwsgi_temp_path {{ deploy_dir }}/uwsgi_temp;
    scgi_temp_path {{ deploy_dir }}/scgi_temp;
    server {
        

## 4. Read the playbook as a state declaration

The playbook links the inventory, variables, and templates. Read its tasks in three groups: **prepare** packages and directories, **render** managed files, and **run** the backend and proxy. On Colab it installs Nginx, creates an application venv, and installs Flask and Gunicorn. Locally those packages come from `uv`. In both modes Ansible renders the same files and manages the same request path.

The process checks report `ok` without claiming a change. Gunicorn starts only when absent or when its application changes; Nginx starts when absent and reloads after a configuration change. PID files make this example work where `systemd` is unavailable. Scan the printed playbook for the `template`, `changed_when: false`, and `when` statements that implement those decisions.

In [8]:
playbook_yaml = """
---
- name: Deploy the DD2482 web service
  hosts: tutorial
  gather_facts: false
  vars_files:
    - vars.yml
  vars:
    deploy_dir: "{{ tutorial_workdir }}/deployed"
    venv_dir: "{{ tutorial_venv }}"
  tasks:
    - name: Verify required OS packages
      ansible.builtin.apt:
        name:
          - nginx
          - python3-venv
        state: present
        update_cache: true
        cache_valid_time: 3600
      when: manage_os_packages

    - name: Ensure deployment directory exists
      ansible.builtin.file:
        path: "{{ deploy_dir }}"
        state: directory
        mode: "0755"

    - name: Create Python virtual environment
      ansible.builtin.command:
        cmd: "python3 -m venv {{ venv_dir }}"
        creates: "{{ venv_dir }}/bin/python"
      when: manage_os_packages

    # Check mode cannot install into a venv that it has not created.
    - name: Install application packages
      ansible.builtin.pip:
        name:
          - Flask
          - gunicorn
        executable: "{{ venv_dir }}/bin/pip"
        state: present
      register: app_packages
      when: manage_os_packages and not ansible_check_mode

    - name: Render Flask application
      ansible.builtin.template:
        src: templates/app.py.j2
        dest: "{{ deploy_dir }}/app.py"
        mode: "0644"
      register: app_template

    - name: Render application configuration
      ansible.builtin.template:
        src: templates/config.json.j2
        dest: "{{ deploy_dir }}/config.json"
        mode: "0644"

    - name: Render Nginx configuration
      ansible.builtin.template:
        src: templates/nginx.conf.j2
        dest: "{{ deploy_dir }}/nginx.conf"
        mode: "0644"
      register: nginx_template

    # Read-only probes avoid reporting a change on every playbook run.
    - name: Check Gunicorn process
      ansible.builtin.shell: >-
        test -f {{ deploy_dir }}/gunicorn.pid &&
        kill -0 "$(cat {{ deploy_dir }}/gunicorn.pid)"
      register: gunicorn_probe
      changed_when: false
      failed_when: false
      check_mode: false
      when: not ansible_check_mode

    # Flask rereads config.json per request, so only code/package changes need a restart.
    - name: Stop Gunicorn after application or package change
      ansible.builtin.shell: "kill $(cat {{ deploy_dir }}/gunicorn.pid)"
      when:
        - not ansible_check_mode
        - gunicorn_probe.rc | default(1) == 0
        - app_template.changed or app_packages.changed | default(false)

    - name: Wait for old backend to stop
      ansible.builtin.wait_for:
        host: 127.0.0.1
        port: "{{ backend_port }}"
        state: stopped
        timeout: 20
      when:
        - not ansible_check_mode
        - gunicorn_probe.rc | default(1) == 0
        - app_template.changed or app_packages.changed | default(false)

    - name: Start Gunicorn when needed
      ansible.builtin.shell: >-
        nohup {{ venv_dir }}/bin/gunicorn
        --bind 127.0.0.1:{{ backend_port }} --workers 1
        --chdir {{ deploy_dir }} app:app
        > {{ deploy_dir }}/gunicorn.log 2>&1 < /dev/null &
        echo $! > {{ deploy_dir }}/gunicorn.pid
      when:
        - not ansible_check_mode
        - gunicorn_probe.rc | default(1) != 0 or app_template.changed or app_packages.changed | default(false)

    - name: Wait for backend
      ansible.builtin.wait_for:
        host: 127.0.0.1
        port: "{{ backend_port }}"
        timeout: 30
      when: not ansible_check_mode

    - name: Validate Nginx configuration
      ansible.builtin.command: "{{ nginx_binary_path }} -t -c {{ deploy_dir }}/nginx.conf"
      changed_when: false
      when: not ansible_check_mode

    - name: Check Nginx process
      ansible.builtin.shell: >-
        test -f {{ deploy_dir }}/nginx.pid &&
        kill -0 "$(cat {{ deploy_dir }}/nginx.pid)"
      register: nginx_probe
      changed_when: false
      failed_when: false
      check_mode: false
      when: not ansible_check_mode

    - name: Start Nginx when needed
      ansible.builtin.command: "{{ nginx_binary_path }} -c {{ deploy_dir }}/nginx.conf"
      when:
        - not ansible_check_mode
        - nginx_probe.rc | default(1) != 0

    - name: Reload Nginx after configuration change
      ansible.builtin.command: "{{ nginx_binary_path }} -s reload -c {{ deploy_dir }}/nginx.conf"
      when:
        - not ansible_check_mode
        - nginx_probe.rc | default(1) == 0
        - nginx_template.changed

    - name: Wait for public endpoint
      ansible.builtin.wait_for:
        host: 127.0.0.1
        port: "{{ public_port }}"
        timeout: 30
      when: not ansible_check_mode
"""
write_source('playbook.yml', playbook_yaml)
print((WORK / 'playbook.yml').read_text())


/tmp/dd2482-ansible-tutorial-2465940/playbook.yml
---
- name: Deploy the DD2482 web service
  hosts: colab
  gather_facts: false
  vars_files:
    - vars.yml
  vars:
    deploy_dir: "{{ tutorial_workdir }}/deployed"
    venv_dir: "{{ tutorial_venv }}"
  tasks:
    - name: Verify required OS packages
      ansible.builtin.apt:
        name:
          - nginx
          - python3-venv
        state: present
      when: manage_os_packages

    - name: Ensure deployment directory exists
      ansible.builtin.file:
        path: "{{ deploy_dir }}"
        state: directory
        mode: "0755"

    - name: Inspect Python virtual environment
      ansible.builtin.stat:
        path: "{{ venv_dir }}/bin/python"
      register: venv_python

    - name: Create Python virtual environment
      ansible.builtin.command:
        cmd: "python3 -m venv {{ venv_dir }}"
        creates: "{{ venv_dir }}/bin/python"
      when: manage_os_packages

    - name: Install application packages
      ansible.buil

## 5. Preview before changing the service

First, `--syntax-check` tests whether Ansible can parse the playbook. Then `--check --diff` predicts changes to managed files without applying them. Expect the preview to mention files that do not exist yet and to show proposed content. Tasks needing a running service are skipped in check mode.

A preview is a review aid, not a health check: it cannot prove that Gunicorn starts or that Nginx forwards requests. That is why the next steps apply the declaration and test the HTTP endpoint. Compare the preview with the real recap after deployment.

In [ ]:
import re

def run_ansible(*arguments):
    command = [ANSIBLE_PLAYBOOK, '-i', 'inventory.ini', *arguments, 'playbook.yml']
    ansible_tmp = WORK / 'ansible-tmp'
    ansible_tmp.mkdir(exist_ok=True)
    ansible_env = dict(os.environ, ANSIBLE_HOME=str(WORK / 'ansible-home'),
                       ANSIBLE_LOCAL_TEMP=str(ansible_tmp),
                       ANSIBLE_REMOTE_TEMP=str(ansible_tmp))
    result = subprocess.run(command, cwd=WORK, env=ansible_env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(f'Ansible exited with status {result.returncode}')
    return result.stdout

run_ansible('--syntax-check')
run_ansible('--check', '--diff')



playbook: playbook.yml



## 6. Apply the desired state

Now Ansible creates the state described above. In the recap, `changed` counts tasks that modified the host, while `ok` counts tasks that found or checked the expected state. The first run should report at least one change because the rendered files and tutorial processes are new.

After the cell finishes, identify which tasks changed. The recap tells you that work happened, but it does not yet prove that a client can reach the application. The next cell supplies that evidence.

In [ ]:
first_recap = run_ansible()
assert re.search(r'changed=[1-9]\d*', first_recap), 'Expected the first deployment to change resources.'

for command in ([str(APP_VENV / 'bin/python'), '-c',
                 'import importlib.metadata as m; print("Flask", m.version("Flask"))'],
                [str(APP_VENV / 'bin/gunicorn'), '--version']):
    print(subprocess.run(command, text=True, capture_output=True, check=True).stdout.strip())


### Checkpoint: declared state became a running service

The first recap should show changes, and the version check above reports the installed Flask and Gunicorn versions. Before trusting the deployment, ask what the recap cannot reveal: it does not tell you whether an HTTP request can travel through Nginx to Flask. The next test answers that question.

## 7. Verify the request path

The test sends `GET /` to Nginx's selected public port. It checks HTTP `200`, the proxy header, and the exact JSON body. The expected body is:

```json
{"application": "dd2482-demo", "version": "1.0", "environment": "tutorial"}
```

The header shows that Nginx handled the response; the body shows that Flask read the rendered configuration. If either component or the connection between them is broken, this cell fails visibly. This is stronger evidence than a successful Ansible command alone.

In [ ]:
import json
import urllib.request

def verify(expected_version):
    with urllib.request.urlopen(f'http://127.0.0.1:{PUBLIC_PORT}/', timeout=10) as response:
        assert response.status == 200
        assert response.headers.get('X-DD2482-Proxy') == 'nginx'
        body = json.load(response)
    expected = {'application': 'dd2482-demo', 'version': expected_version,
                'environment': 'tutorial'}
    assert body == expected, f'Expected {expected}, got {body}'
    print('HTTP 200 via Nginx:', body)

verify('1.0')


## 8. Prove idempotency

Run the same playbook again **without editing any source file**. Ansible should find the declared state already present, and the recap should report `changed=0`. The assertion makes an unexpected change fail the cell.

This result demonstrates idempotency for the resources in this example: repeated application converges to the same state. Keep the HTTP result from the previous step in view as well; a zero-change recap by itself says nothing about whether the service answers correctly.

In [ ]:
second_recap = run_ansible()
changes = re.findall(r'changed=(\d+)', second_recap)
assert changes and int(changes[-1]) == 0, f'Expected changed=0, got {changes}'
print('Idempotency confirmed: changed=0')


### Checkpoint: the host and declaration agree

The service returned the declared JSON, and the repeat run changed nothing. This is the baseline for the incident experiment. Next, you will change the host while keeping the declaration fixed, then compare the two again.

## 9. Create configuration drift

Now act like the teammate from the opening scenario. Edit the **deployed** `config.json` directly so its version becomes `manual-edit`. Do not change `vars.yml` or the Jinja source. Because Flask reads the JSON file on each request, the next response should show the manual value immediately.

Before running the cell, predict the two values: what does the repository still declare, and what will the service report? Their disagreement is configuration drift. The application may be healthy in the HTTP sense while its configuration is wrong.

In [ ]:
config_path = DEPLOY / 'config.json'
actual = json.loads(config_path.read_text())
actual['version'] = 'manual-edit'
config_path.write_text(json.dumps(actual, indent=2) + '\n')
with urllib.request.urlopen(f'http://127.0.0.1:{PUBLIC_PORT}/', timeout=10) as response:
    drifted = json.load(response)
assert drifted['version'] == 'manual-edit'
print('Drifted response:', drifted)


## 10. Detect and repair the drift

Run `--check --diff` again. This time the interesting change is in `config.json`: the deployed `manual-edit` value should be replaced with the declared `1.0`. The notebook checks that both values appear in the preview, then applies the playbook and tests the endpoint again.

Inspect the repair recap. Ideally only the managed configuration file changes; the application and proxy do not need a restart because the application reads that file for each request. The final HTTP assertion closes the loop: the service reports `1.0` again.

In [ ]:
drift_preview = run_ansible('--check', '--diff')
assert 'manual-edit' in drift_preview and '1.0' in drift_preview, 'Expected config diff was not shown.'
repair_recap = run_ansible()
assert re.search(r'changed=[1-9]\d*', repair_recap), 'Expected Ansible to repair drift.'
verify('1.0')


### Checkpoint: the declaration repaired the host

The preview identified the difference, and the apply step restored the response to `1.0`. Notice that repair did not require reconstructing the entire VM: Ansible compared managed resources and corrected the one that drifted. The next step tests the opposite direction—changing the declaration on purpose.

## 11. Perform a controlled update

The drift was an unrecorded edit to the host. A release update starts in the declaration. This cell changes `app_version` in `vars.yml` from `1.0` to `2.0`, previews the file diff, applies it, and verifies the new HTTP response. It then reapplies the playbook and expects `changed=0` again.

Watch the order: **edit source → preview → apply → verify → repeat**. The same Ansible mechanism handles both repair and planned change; the difference is whether the source of truth was deliberately updated first.

In [ ]:
vars_path = WORK / 'vars.yml'
variables = vars_path.read_text()
if 'app_version: "1.0"' in variables:
    vars_path.write_text(variables.replace('app_version: "1.0"', 'app_version: "2.0"'))
else:
    assert 'app_version: "2.0"' in variables, 'Unexpected version in vars.yml'
update_preview = run_ansible('--check', '--diff')
assert '2.0' in update_preview, 'Expected the proposed version in the preview.'
run_ansible()
verify('2.0')
final_recap = run_ansible()
final_changes = re.findall(r'changed=(\d+)', final_recap)
assert final_changes and int(final_changes[-1]) == 0
print('Updated state is idempotent: changed=0')


### Checkpoint: a reviewed release reached the service

The declared version and HTTP response now agree at `2.0`, and the repeat run is again idempotent. Compare this with `manual-edit`: both changed the visible response, but only the `2.0` update was recorded in `vars.yml` and previewed before deployment.

## 12. Reflect on the design

You have seen three distinct observations: a healthy service, a zero-change Ansible run, and a diff that exposed and repaired drift. Together they tell a stronger story than any one observation alone. Consider these questions before reading the notes below:

1. Why did editing the deployed JSON change the response without changing `vars.yml`?
2. Which task should report `changed` during repair, and why do later runs return to zero?
3. If the playbook completed but the HTTP assertion failed, where in the request path would you investigate first?

**Design notes.** Ansible is useful when hosts exist and their packages, files, and processes need configuration. Inventory separates the target from the playbook; variables and templates avoid duplicating release values. Testing through Nginx checks the interaction between proxy and application, rather than assuming a successful playbook means the service works.

This is a single-host teaching example. It has no redundancy, durable storage, or remote hosts. PID files keep it runnable in Colab and local environments without `systemd`; production services would normally use a supervisor and stronger health checks. A remote inventory would use hostnames and SSH access, with credentials managed outside the playbook. Terraform can provision networks and VMs, while Ansible can configure the software on them.

## 13. Clean up

The experiment is complete. The final cell stops Gunicorn and Nginx, then removes the temporary working directory and the deployed configuration. Run it so the chosen ports become available again. Packages remain in the disposable Colab VM or the local `.venv`, so you can repeat the exercise without rebuilding the development environment.

In [ ]:
import signal
import time

nginx_config = DEPLOY / 'nginx.conf'
if pid_is_alive(DEPLOY / 'nginx.pid') and nginx_config.exists():
    subprocess.run([NGINX_BINARY, '-s', 'stop', '-c', str(nginx_config)], check=True)
if pid_is_alive(DEPLOY / 'gunicorn.pid'):
    os.kill(int((DEPLOY / 'gunicorn.pid').read_text().strip()), signal.SIGTERM)
for pid_file in (DEPLOY / 'nginx.pid', DEPLOY / 'gunicorn.pid'):
    for _ in range(50):
        if not pid_is_alive(pid_file):
            break
        time.sleep(0.1)
shutil.rmtree(WORK)
print('Tutorial services stopped and working directory removed.')
